In [1]:
import pandas as pd
import numpy as np
import json
import os
import plotly.express as px

# --- CONFIGURATION ---
INPUT_DIR = "/kaggle/input/datasets/mianafzalsaeed/nlp-mp-pipeline-all-results" 
OUTPUT_DIR = "/kaggle/working"

# Outlier Thresholds for T4 GPUs
MAX_REALISTIC_TPS = 150.0  
MIN_REALISTIC_TTFT = 0.1

FILES_MAPPING = {
    ("Naive", "3B", "28x28"): "MASTER_3B_EXTREME_28x28_with_naive_approach.json",
    ("Standard", "3B", "28x28"): "MASTER_3B_EXTREME_Results.json",
    ("Naive", "3B", "224x224"): "MASTER_3B_STANDARD_224x224_with_naive_approach.json",
    ("Standard", "3B", "224x224"): "MASTER_3B_STANDARD_Results.json",
    ("Naive", "7B", "28x28"): "MASTER_7B_EXTREME_28x28_with_naive_approach.json",
    ("Standard", "7B", "28x28"): "MASTER_7B_EXTREME_Results.json",
    ("Naive", "7B", "224x224"): "MASTER_7B_STANDARD_224x224_with_naive_approach.json",
    ("Standard", "7B", "224x224"): "MASTER_7B_STANDARD_Results.json"
}

MVBENCH_TASKS = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

def get_task_name(index):
    task_idx = index // 200
    if 0 <= task_idx < len(MVBENCH_TASKS):
        return MVBENCH_TASKS[task_idx]
    return "unknown_task"

print("Starting Data Compilation & Artifact Scrubbing...")

# 1. EXTRACT TIDY DATA
tidy_data = []

for (approach, model, res), filename in FILES_MAPPING.items():
    filepath = os.path.join(INPUT_DIR, filename)
    if not os.path.exists(filepath):
        filepath = os.path.join(OUTPUT_DIR, filename)
        if not os.path.exists(filepath): continue
        
    with open(filepath, "r") as f:
        records = json.load(f)
    
    temp_metrics = {task: {"correct": 0, "total": 0, "tps_sum": 0.0, "tps_count": 0, "ttft_sum": 0.0, "ttft_count": 0} for task in MVBENCH_TASKS}
    
    for r in records:
        task = get_task_name(r["id"])
        
        # Acc
        gt = r.get("ground_truth_letter", "").strip().upper()
        pred = r.get("response", "").strip().upper()
        if gt and pred == gt: temp_metrics[task]["correct"] += 1
        temp_metrics[task]["total"] += 1
        
        # TPS
        tps = r.get("tokens_per_second", 0.0)
        if tps > 0:
            temp_metrics[task]["tps_sum"] += tps
            temp_metrics[task]["tps_count"] += 1
            
        # TTFT
        ttft = r.get("ttft_seconds", 0.0)
        if ttft > 0:
            temp_metrics[task]["ttft_sum"] += ttft
            temp_metrics[task]["ttft_count"] += 1

    for task in MVBENCH_TASKS:
        total = temp_metrics[task]["total"]
        tps_c = temp_metrics[task]["tps_count"]
        ttft_c = temp_metrics[task]["ttft_count"]
        
        acc = (temp_metrics[task]["correct"] / total * 100) if total > 0 else np.nan
        avg_tps = (temp_metrics[task]["tps_sum"] / tps_c) if tps_c > 0 else np.nan
        avg_ttft = (temp_metrics[task]["ttft_sum"] / ttft_c) if ttft_c > 0 else np.nan
        
        tidy_data.append({
            "Task": task, "Tokenization": approach, "Model": model, 
            "Resolution": res, "Accuracy": acc, "TPS": avg_tps, "TTFT": avg_ttft
        })

df = pd.DataFrame(tidy_data)

# --- APPLY OUTLIER TRUNCATION (SCRUBBING) ---
# Replace physically impossible math artifacts with NaN so they do not ruin the averages or charts
initial_tps_count = df['TPS'].notna().sum()
df.loc[df['TPS'] > MAX_REALISTIC_TPS, 'TPS'] = np.nan
df.loc[df['TTFT'] < MIN_REALISTIC_TTFT, 'TTFT'] = np.nan

print(f"[*] Scrubbed {initial_tps_count - df['TPS'].notna().sum()} extreme TPS math artifacts from the dataset.")

# 2. CREATE 2D TABLES (PIVOTS)
def generate_2d_table(dataframe, metric, aggfunc='mean'):
    pivot = pd.pivot_table(
        dataframe, values=metric, index=['Model'], 
        columns=['Tokenization', 'Resolution'], aggfunc=aggfunc
    ).round(2)
    return pivot

html_content = "<html><head><style>table {border-collapse: collapse; margin-bottom: 30px;} th, td {border: 1px solid black; padding: 8px; text-align: center;} th {background-color: #f2f2f2;}</style></head><body>"
html_content += "<h1>Ablation Study: 2D Results Tables (Artifacts Scrubbed)</h1>"

html_content += "<h2>High-Level Averages</h2>"
for metric in ['Accuracy', 'TPS', 'TTFT']:
    html_content += f"<h3>Average {metric}</h3>"
    html_content += generate_2d_table(df, metric, 'mean').to_html(na_rep="")
    
html_content += "<h2>Per-Task Breakdowns</h2>"
for task in MVBENCH_TASKS:
    task_df = df[df['Task'] == task]
    if task_df.dropna(subset=['Accuracy']).empty: continue
    html_content += f"<h3>Task: {task}</h3>"
    for metric in ['Accuracy', 'TPS', 'TTFT']:
        html_content += f"<h4>{metric}</h4>"
        html_content += generate_2d_table(task_df, metric, 'mean').to_html(na_rep="")

html_content += "</body></html>"

# Save Clean 2D Tables Report
report_path = os.path.join(OUTPUT_DIR, "1_2D_TABLES_REPORT.html")
with open(report_path, "w") as f:
    f.write(html_content)
print(f"[*] 2D Tables Report saved -> {report_path}")

# 3. INTERACTIVE VISUALIZATIONS
print("Generating Cleaned Interactive Visualizations...")

plot_df = df.copy()
plot_df['Config'] = plot_df['Tokenization'] + "_" + plot_df['Model'] + "_" + plot_df['Resolution']

fig_acc = px.bar(plot_df, x='Task', y='Accuracy', color='Config', barmode='group', title='Interactive Accuracy Comparison')
fig_acc.write_html(os.path.join(OUTPUT_DIR, "2_INTERACTIVE_ACCURACY.html"))

# This chart will now look perfect, with no massive bars ruining the scale
fig_tps = px.bar(plot_df, x='Task', y='TPS', color='Config', barmode='group', title='Interactive Decode Speed (TPS) - Truncated')
fig_tps.write_html(os.path.join(OUTPUT_DIR, "3_INTERACTIVE_TPS.html"))

fig_ttft = px.bar(plot_df, x='Task', y='TTFT', color='Config', barmode='group', title='Interactive Time-To-First-Token (TTFT)')
fig_ttft.write_html(os.path.join(OUTPUT_DIR, "4_INTERACTIVE_TTFT.html"))

print("\nSuccess! Generated all scrubbed files in your /kaggle/working/ directory.")

Starting Data Compilation & Artifact Scrubbing...
[*] Scrubbed 12 extreme TPS math artifacts from the dataset.
[*] 2D Tables Report saved -> /kaggle/working/1_2D_TABLES_REPORT.html
Generating Cleaned Interactive Visualizations...

Success! Generated all scrubbed files in your /kaggle/working/ directory.


In [2]:
import pandas as pd
import numpy as np
import json
import os

# --- CONFIGURATION ---
INPUT_DIR = "/kaggle/input/datasets/mianafzalsaeed/nlp-mp-pipeline-all-results" 
OUTPUT_DIR = "/kaggle/working"

# Outlier Thresholds (to fix the 600+ TPS artifacts)
MAX_REALISTIC_TPS = 150.0  
MIN_REALISTIC_TTFT = 0.1

FILES_MAPPING = {
    ("Naive", "3B", "28x28"): "MASTER_3B_EXTREME_28x28_with_naive_approach.json",
    ("Standard", "3B", "28x28"): "MASTER_3B_EXTREME_Results.json",
    ("Naive", "3B", "224x224"): "MASTER_3B_STANDARD_224x224_with_naive_approach.json",
    ("Standard", "3B", "224x224"): "MASTER_3B_STANDARD_Results.json",
    ("Naive", "7B", "28x28"): "MASTER_7B_EXTREME_28x28_with_naive_approach.json",
    ("Standard", "7B", "28x28"): "MASTER_7B_EXTREME_Results.json",
    ("Naive", "7B", "224x224"): "MASTER_7B_STANDARD_224x224_with_naive_approach.json",
    ("Standard", "7B", "224x224"): "MASTER_7B_STANDARD_Results.json"
}

MVBENCH_TASKS = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

def get_task_name(index):
    task_idx = index // 200
    if 0 <= task_idx < len(MVBENCH_TASKS):
        return MVBENCH_TASKS[task_idx]
    return "unknown_task"

print("Parsing JSON files and aggregating data...")

# 1. EXTRACT DATA
tidy_data = []

for (approach, model, res), filename in FILES_MAPPING.items():
    # Try input dir first, fallback to working dir
    filepath = os.path.join(INPUT_DIR, filename)
    if not os.path.exists(filepath):
        filepath = os.path.join(OUTPUT_DIR, filename)
        if not os.path.exists(filepath):
            print(f" [!] Warning: {filename} not found. Skipping.")
            continue
        
    with open(filepath, "r") as f:
        records = json.load(f)
    
    temp_metrics = {task: {"correct": 0, "total": 0, "tps_sum": 0.0, "tps_count": 0, "ttft_sum": 0.0, "ttft_count": 0} for task in MVBENCH_TASKS}
    
    for r in records:
        task = get_task_name(r["id"])
        
        # Accuracy Check
        gt = r.get("ground_truth_letter", "").strip().upper()
        pred = r.get("response", "").strip().upper()
        if gt and pred == gt: temp_metrics[task]["correct"] += 1
        temp_metrics[task]["total"] += 1
        
        # TPS Check
        tps = r.get("tokens_per_second", 0.0)
        if tps > 0:
            temp_metrics[task]["tps_sum"] += tps
            temp_metrics[task]["tps_count"] += 1
            
        # TTFT Check
        ttft = r.get("ttft_seconds", 0.0)
        if ttft > 0:
            temp_metrics[task]["ttft_sum"] += ttft
            temp_metrics[task]["ttft_count"] += 1

    # Calculate averages per task
    for task in MVBENCH_TASKS:
        total = temp_metrics[task]["total"]
        tps_c = temp_metrics[task]["tps_count"]
        ttft_c = temp_metrics[task]["ttft_count"]
        
        acc = (temp_metrics[task]["correct"] / total * 100) if total > 0 else np.nan
        avg_tps = (temp_metrics[task]["tps_sum"] / tps_c) if tps_c > 0 else np.nan
        avg_ttft = (temp_metrics[task]["ttft_sum"] / ttft_c) if ttft_c > 0 else np.nan
        
        tidy_data.append({
            "Task": task, "Tokenization": approach, "Model": model, 
            "Resolution": res, "Accuracy": acc, "TPS": avg_tps, "TTFT": avg_ttft
        })

# Create the main Combined DataFrame
df_combined = pd.DataFrame(tidy_data)

# --- APPLY OUTLIER TRUNCATION (SCRUBBING) ---
df_combined.loc[df_combined['TPS'] > MAX_REALISTIC_TPS, 'TPS'] = np.nan
df_combined.loc[df_combined['TTFT'] < MIN_REALISTIC_TTFT, 'TTFT'] = np.nan

# --- SEPARATE THE DATA ---
print("Splitting datasets by Tokenization approach...")
df_naive = df_combined[df_combined['Tokenization'] == 'Naive'].copy()
df_standard = df_combined[df_combined['Tokenization'] == 'Standard'].copy()

# (Optional) Drop the 'Tokenization' column from the separate files since it's redundant
df_naive = df_naive.drop(columns=['Tokenization'])
df_standard = df_standard.drop(columns=['Tokenization'])

# --- EXPORT TO CSV ---
combined_path = os.path.join(OUTPUT_DIR, "COMBINED_RESULTS.csv")
naive_path = os.path.join(OUTPUT_DIR, "NAIVE_RESULTS_ONLY.csv")
standard_path = os.path.join(OUTPUT_DIR, "STANDARD_RESULTS_ONLY.csv")

# Format NaNs as blanks ("") for clean Excel viewing
df_combined.fillna("").to_csv(combined_path, index=False)
df_naive.fillna("").to_csv(naive_path, index=False)
df_standard.fillna("").to_csv(standard_path, index=False)

print("\nSuccess! Files exported to:")
print(f" 1. {combined_path}")
print(f" 2. {naive_path}")
print(f" 3. {standard_path}")

Parsing JSON files and aggregating data...
Splitting datasets by Tokenization approach...

Success! Files exported to:
 1. /kaggle/working/COMBINED_RESULTS.csv
 2. /kaggle/working/NAIVE_RESULTS_ONLY.csv
 3. /kaggle/working/STANDARD_RESULTS_ONLY.csv
